# Phase 2D — Regression Analysis

**Theory to study:** Simple linear regression, multiple regression, OLS, R², residuals, multicollinearity.

**Tools used:** `numpy`, `statsmodels` (full statistical output), `sklearn` (ML-style preview), `matplotlib`

**Install if needed:**
```
pip install statsmodels scikit-learn
```

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---
## 1. Simple Linear Regression — From Scratch with NumPy

**Model:** `y = β₀ + β₁x + ε`

- `β₀` = intercept (y when x=0)
- `β₁` = slope (change in y per unit change in x)
- `ε` = error term (residuals)

In [ ]:
# Dataset: house size (sq ft) vs price ($1000s)
house_size = np.array(
    [
        750,
        850,
        1000,
        1100,
        1200,
        1350,
        1500,
        1600,
        1750,
        2000,
        2100,
        2300,
        2500,
        2700,
        3000,
    ]
)
house_price = np.array(
    [150, 175, 195, 210, 220, 250, 280, 295, 330, 375, 385, 420, 460, 495, 550]
)

# OLS formulas
n = len(house_size)
x_bar = house_size.mean()
y_bar = house_price.mean()

beta_1 = np.sum((house_size - x_bar) * (house_price - y_bar)) / np.sum(
    (house_size - x_bar) ** 2
)
beta_0 = y_bar - beta_1 * x_bar

print(f"Intercept (β₀): {beta_0:.4f}")
print(
    f"Slope     (β₁): {beta_1:.6f}  (each extra sq ft adds ${beta_1 * 1000:.2f} to price)"
)

In [ ]:
# Predictions and residuals
y_pred = beta_0 + beta_1 * house_size
residuals = house_price - y_pred

# R² — coefficient of determination
ss_res = np.sum(residuals**2)  # residual sum of squares
ss_tot = np.sum((house_price - y_bar) ** 2)  # total sum of squares
r_squared = 1 - (ss_res / ss_tot)

# RMSE
rmse = np.sqrt(ss_res / n)

print(
    f"R²   : {r_squared:.4f}  ({r_squared * 100:.1f}% of variance explained by house size)"
)
print(f"RMSE : {rmse:.2f}  ($1000s average prediction error)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Regression line
x_line = np.linspace(700, 3100, 200)
axes[0].scatter(
    house_size, house_price, color="steelblue", s=70, label="Actual data", zorder=3
)
axes[0].plot(
    x_line,
    beta_0 + beta_1 * x_line,
    "r-",
    lw=2,
    label=f"ŷ = {beta_0:.0f} + {beta_1:.4f}x  (R²={r_squared:.3f})",
)
axes[0].set_title("Simple Linear Regression")
axes[0].set_xlabel("House Size (sq ft)")
axes[0].set_ylabel("Price ($1000s)")
axes[0].legend()

# Residual plot — should be random scatter (no pattern)
axes[1].scatter(y_pred, residuals, color="coral", s=70)
axes[1].axhline(0, color="black", linestyle="--", lw=1)
axes[1].set_title("Residual Plot")
axes[1].set_xlabel("Fitted Values")
axes[1].set_ylabel("Residuals")

plt.tight_layout()
plt.show()

---
## 2. Numpy `polyfit` — Quick Linear Fit

In [ ]:
# np.polyfit fits a polynomial of degree 1 (line)
coeffs = np.polyfit(house_size, house_price, deg=1)
slope, intercept = coeffs
print(f"np.polyfit → slope={slope:.6f}, intercept={intercept:.4f}")

# Create a poly1d object for easy evaluation
poly_model = np.poly1d(coeffs)
print(f"Predicted price for 2000 sq ft: ${poly_model(2000):.1f}k")
print(f"Predicted price for 2500 sq ft: ${poly_model(2500):.1f}k")

---
## 3. statsmodels OLS — Full Statistical Output

`statsmodels` gives you the full statistical summary: coefficients, p-values, confidence intervals, F-statistic — everything you need to interpret a regression properly.

In [ ]:
try:
    import statsmodels.api as sm

    # statsmodels does NOT add an intercept automatically — we must add it
    X = sm.add_constant(house_size)  # adds a column of 1s for the intercept
    model = sm.OLS(house_price, X).fit()

    print(model.summary())
except ImportError:
    print("statsmodels not installed. Run: pip install statsmodels")
    print()
    print("Key things the summary() table tells you:")
    print("  R-squared   — proportion of variance explained")
    print("  Adj. R²     — R² penalized for number of predictors")
    print("  F-statistic — tests if the overall model is significant")
    print("  coef        — estimated β coefficients")
    print("  P>|t|       — p-value for each coefficient (is it significant?)")
    print("  [0.025 0.975] — 95% confidence interval for each coefficient")

---
## 4. Multiple Linear Regression

**Model:** `y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ + ε`

Multiple predictors allow us to control for confounding variables.

In [ ]:
# Predict house price using: size, bedrooms, age of house
np.random.seed(42)
n_samples = 100

size = np.random.uniform(800, 3500, n_samples)
bedrooms = np.random.randint(2, 6, n_samples).astype(float)
age = np.random.uniform(0, 50, n_samples)

# True relationship (plus noise)
price = (
    50 + 0.15 * size + 20 * bedrooms - 1.5 * age + np.random.normal(0, 20, n_samples)
)

try:
    import statsmodels.api as sm

    X_multi = sm.add_constant(np.column_stack([size, bedrooms, age]))
    model_multi = sm.OLS(price, X_multi).fit()

    print(model_multi.summary())
except ImportError:
    # Fallback: use numpy least squares
    X_multi = np.column_stack([np.ones(n_samples), size, bedrooms, age])
    coeffs_multi, _, _, _ = np.linalg.lstsq(X_multi, price, rcond=None)

    intercept, b_size, b_bedrooms, b_age = coeffs_multi
    print(f"Intercept  : {intercept:.2f}")
    print(f"Size       : {b_size:.4f}  (each +1 sq ft → +${b_size * 1000:.2f})")
    print(
        f"Bedrooms   : {b_bedrooms:.2f}  (each extra bedroom → +${b_bedrooms * 1000:.0f})"
    )
    print(f"Age        : {b_age:.4f}  (each year older → {b_age * 1000:+.0f} in price)")

    y_hat = X_multi @ coeffs_multi
    ss_res = np.sum((price - y_hat) ** 2)
    ss_tot = np.sum((price - price.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    print(f"\nR² (numpy): {r2:.4f}")

---
## 5. Residual Diagnostics

OLS assumes:
1. **Linearity** — residuals have no pattern vs fitted values
2. **Normality** — residuals are normally distributed
3. **Homoscedasticity** — residuals have constant variance
4. **Independence** — residuals are not autocorrelated

In [ ]:
# Using the simple regression residuals from Section 1
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Residuals vs Fitted
axes[0].scatter(y_pred, residuals, color="steelblue", s=60)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals vs Fitted")
axes[0].set_xlabel("Fitted Values")
axes[0].set_ylabel("Residuals")

# 2. Q-Q plot (normality of residuals)
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot of Residuals")

# 3. Histogram of residuals
axes[2].hist(residuals, bins=8, color="steelblue", edgecolor="white", density=True)
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
axes[2].plot(
    x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()), "r-", lw=2
)
axes[2].set_title("Distribution of Residuals")
axes[2].set_xlabel("Residual")

plt.tight_layout()
plt.show()

---
## 6. Preview — scikit-learn LinearRegression

`sklearn` is the primary ML library. Its API is: **fit → predict → score**. No automatic intercept addition needed.

In [ ]:
try:
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import r2_score, mean_squared_error

    # sklearn expects 2D X array
    X_sk = house_size.reshape(-1, 1)  # shape (15, 1)
    y_sk = house_price

    model_sk = LinearRegression()
    model_sk.fit(X_sk, y_sk)

    print(f"sklearn — Intercept: {model_sk.intercept_:.4f}")
    print(f"sklearn — Slope    : {model_sk.coef_[0]:.6f}")

    y_pred_sk = model_sk.predict(X_sk)
    print(f"R²   : {r2_score(y_sk, y_pred_sk):.4f}")
    print(f"RMSE : {mean_squared_error(y_sk, y_pred_sk) ** 0.5:.4f}")

    # Predict on new data
    new_sizes = np.array([[1500], [2000], [2500]])
    predictions = model_sk.predict(new_sizes)
    for size_val, pred in zip(new_sizes.flatten(), predictions):
        print(f"  House {size_val} sq ft → predicted ${pred:.1f}k")

except ImportError:
    print("scikit-learn not installed. Run: pip install scikit-learn")
    print("We will use it extensively in Phase 5.")
    print()
    print("sklearn API pattern (same for ALL models):")
    print("  model = LinearRegression()")
    print("  model.fit(X_train, y_train)")
    print("  y_pred = model.predict(X_test)")
    print("  score  = model.score(X_test, y_test)")

---
## Summary

| Tool | Best For |
|------|----------|
| `np.polyfit` | Quick 1-variable fit, no statistical output |
| `np.linalg.lstsq` | Matrix-form OLS, no statistical output |
| `statsmodels.OLS` | Full statistical output (p-values, CI, F-stat) — use for analysis |
| `sklearn.LinearRegression` | ML pipelines, predictions, model comparison — use for production |

**Key metrics:**
- **R²**: 0 = model explains nothing, 1 = model explains everything
- **RMSE**: average prediction error in original units
- **Residuals**: check for normality, constant variance, no patterns
- **p-value on coefficients**: is each predictor statistically significant?